In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

In [16]:
import re
import pandas as pd
import numpy as np
import nltk

# Download required NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print('All libraries imported successfully!')

All libraries imported successfully!


In [17]:
def lower_order(text):
    """
    Converts all text to lowercase.
    Input Args:
        text: input string.
    Returns:
        Lowercase version of text.
    """
    return text.lower()

def remove_urls(text):
    """
    Removes URLs from text using regex.
    Input Args:
        text: string that may contain URLs.
    Returns:
        text with URLs replaced by empty string.
    """
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_emoji(string):
    """
    Replaces emojis in a string with whitespace.
    Input Args:
        string: input text.
    Returns:
        text with emojis replaced by space.
    """
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r' ', string)

def removeunwanted_characters(document):
    """
    Removes @mentions, #hashtags, punctuation, emojis, and extra spaces.
    Input Args:
        document: input text.
    Returns:
        Cleaned text string.
    """
    document = re.sub("@[A-Za-z0-9_]+", " ", document)   # remove mentions
    document = re.sub("#[A-Za-z0-9_]+", "", document)     # remove hashtags
    document = re.sub("[^0-9A-Za-z ]", "", document)      # remove punctuation
    document = remove_emoji(document)                       # remove emojis
    document = document.replace('  ', "")                  # remove double spaces
    return document.strip()

stop_words = set(stopwords.words('english'))
custom_stopwords = ['@', 'RT']
stop_words.update(custom_stopwords)

def remove_stopwords(text_tokens):
    """
    Removes stopwords from a list of tokens.
    Input Args:
        text_tokens: list of word tokens.
    Returns:
        List of tokens with stopwords removed.
    """
    return [token for token in text_tokens if token not in stop_words]

def lemmatization(token_text):
    """
    Performs lemmatization on a list of tokens.
    Input Args:
        token_text: list of tokens.
    Returns:
        List of lemmatized tokens.
    """
    wordnet = WordNetLemmatizer()
    return [wordnet.lemmatize(token, pos='v') for token in token_text]

def stemming(text):
    """
    Performs stemming on a list of tokens.
    Input Args:
        text: list of tokens.
    Returns:
        List of stemmed tokens.
    """
    porter = PorterStemmer()
    return [porter.stem(word) for word in text]


print('All helper functions defined!')

All helper functions defined!


# Build a Text Cleaning Pipeline

In [18]:
def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Full text cleaning pipeline:
      1. Lowercases text
      2. Removes URLs
      3. Removes emojis
      4. Removes unwanted characters (@mentions, #hashtags, punctuation)
      5. Tokenizes
      6. Removes stopwords
      7. Applies lemmatization OR stemming based on 'rule'

    Input Args:
        dataset (str): Raw input text.
        rule (str): 'lemmatize' or 'stem'. Default is 'lemmatize'.
    Returns:
        str: Cleaned text as a single joined string.
    """
    # Step 1: Convert to lowercase
    data = lower_order(dataset)

    # Step 2: Remove URLs
    data = remove_urls(data)

    # Step 3: Remove emojis
    data = remove_emoji(data)

    # Step 4: Remove all other unwanted characters
    data = removeunwanted_characters(data)

    # Step 5: Tokenize
    tokens = data.split()

    # Step 6: Remove stopwords
    tokens = remove_stopwords(tokens)

    # Step 7: Lemmatize or Stem
    if rule == "lemmatize":
        tokens = lemmatization(tokens)
    elif rule == "stem":
        tokens = stemming(tokens)
    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)


# Quick test
sample = "Hello @gabe_flomo 👋🏾, still want us to hit that new sushi spot??? LMK when you're free cuz I can't go this or next weekend since I'll be swimming!!! #sushiBros #rawFish #🍱"
print("Original:", sample)
print("Cleaned: ", text_cleaning_pipeline(sample))

Original: Hello @gabe_flomo 👋🏾, still want us to hit that new sushi spot??? LMK when you're free cuz I can't go this or next weekend since I'll be swimming!!! #sushiBros #rawFish #🍱
Cleaned:  hello still want us hit new sushi spot lmk youre free cuz cant go next weekend since ill swim


# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [33]:
# Load the dataset
data = pd.read_csv("/content/drive/MyDrive/AI ML/DATA/trum_tweet_sentiment_analysis.csv", encoding="ISO-8859-1")
print("Dataset shape:", data.shape)
data.head()

Dataset shape: (1850123, 2)


,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [37]:
data_cleaning = data[['text','Sentiment']].dropna()

In [44]:
data_cleaning.iloc[0]

,0
text,RT @JohnLeguizamo: #trump not draining swamp b...
Sentiment,0


In [46]:
X = data_cleaning['text']
y = data_cleaning['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # keeps label distribution balanced across splits
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

Training samples : 1480098
Testing samples  : 370025


In [47]:
# Fit TF-IDF on training data, then transform both train and test
tfidf = TfidfVectorizer(
    max_features=5000,   # keep the top 5000 terms by TF-IDF score
    ngram_range=(1, 2),  # use unigrams and bigrams
    sublinear_tf=True    # apply sublinear TF scaling
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("TF-IDF training matrix shape :", X_train_tfidf.shape)
print("TF-IDF testing  matrix shape :", X_test_tfidf.shape)

TF-IDF training matrix shape : (1480098, 5000)
TF-IDF testing  matrix shape : (370025, 5000)


In [50]:
# Train Logistic Regression
clf = LogisticRegression(
    max_iter=500,
    random_state=42
)
clf.fit(X_train_tfidf, y_train)

# Predict on test set
y_pred = clf.predict(X_test_tfidf)

# Evaluation
print("CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred))

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.90      0.94      0.92    248842
           1       0.86      0.79      0.83    121183

    accuracy                           0.89    370025
   macro avg       0.88      0.86      0.87    370025
weighted avg       0.89      0.89      0.89    370025

